# Taller 6: Ciclos + Funciones (continuación)

Este taller continúa la serie de nivelación (Talleres 1 a 5). Ahí cada problema se resolvía con **un** ciclo dentro de **una** función. Aquí subimos un peldaño: vas a **combinar** varias funciones entre sí (una función que llama a otra dentro de un ciclo) y a **generalizar con parámetros** cosas que antes tenías fijas dentro del código — es decir, vas a descomponer problemas más grandes en varias funciones pequeñas que cooperan.

> **Cómo usar este material:** cada ejercicio trae **enunciado**, **modelo de pensamiento** (el razonamiento antes de programar) y **solución en Python** con type hints. A partir de este taller, cada solución agrega además un breve análisis de **complejidad temporal (tiempo) y espacial (memoria)** en notación Big-O — no basta con que el código funcione, hay que entender qué tan caro es.

## Ejercicio 1: De código duplicado a función con parámetros

### Enunciado

Vas a ver dos funciones casi idénticas: `contar_aprobados(notas)` (cuenta las notas mayores o iguales a 3.0) y `contar_con_honores(notas)` (cuenta las notas mayores o iguales a 4.5). Refactorízalas en una sola función `contar_sobre_umbral(notas, umbral)` que reciba el umbral como parámetro, y úsala para reproducir ambos conteos.

### Modelo de pensamiento

1. Compara las dos funciones línea por línea: el ciclo, el contador y el `if` son idénticos — la única diferencia real es el número fijo (`3.0` frente a `4.5`) usado en la comparación. Esa es la señal de "código duplicado" que ya viste en el Taller 4, ahora con un ciclo dentro.
2. El número que cambiaba entre las dos versiones se convierte en un **parámetro nuevo** de la función: en vez de que el umbral esté "quemado" dentro del código (`nota >= 3.0`), pasa a ser algo que decide quien llama la función (`nota >= umbral`).
3. Con esto, una sola función sirve para ambos casos: `contar_sobre_umbral(notas, 3.0)` reemplaza a `contar_aprobados`, y `contar_sobre_umbral(notas, 4.5)` reemplaza a `contar_con_honores` — cada llamada solo cambia el segundo argumento, sin tocar el cuerpo de la función.

In [1]:
def contar_sobre_umbral(notas: list[float], umbral: float) -> int:
    contador = 0
    for nota in notas:
        if nota >= umbral:
            contador += 1
    return contador


notas = [4.5, 2.0, 3.0, 1.5, 5.0, 4.8]
print(contar_sobre_umbral(notas, 3.0))   # aprobados: 4
print(contar_sobre_umbral(notas, 4.5))   # con honores: 3


4
3


**Complejidad:** un solo recorrido de la lista, con una comparación O(1) por elemento → **O(n) en tiempo**, con `n = len(notas)`. En **espacio** es **O(1)** adicional: solo se guarda el contador, sin estructuras nuevas proporcionales a `n`.

## Ejercicio 2: Validar una contraseña con funciones auxiliares

### Enunciado

Escribe una función `es_valida(contraseña, longitud_minima=8)` que devuelva `True` solo si la contraseña cumple **tres** reglas: tiene al menos `longitud_minima` caracteres, contiene al menos una mayúscula, y contiene al menos un dígito. Implementa cada regla en su propia función auxiliar (`tiene_mayuscula`, `tiene_digito`), recorriendo la cadena carácter por carácter.

### Modelo de pensamiento

1. Cuando un problema mezcla varias reglas independientes, la señal de diseño es **una función por regla** — así cada una se puede probar y leer por separado, en vez de meter todo en un único ciclo con muchos `if` anidados.
2. Cada regla de "¿existe al menos un carácter que cumpla X?" es el patrón de **búsqueda con corte anticipado** que ya viste en el Taller 2: en cuanto encuentras uno que cumple, no hace falta seguir revisando — `return True` inmediatamente. Si el ciclo termina sin encontrar nada, la respuesta es `False`.
3. La función principal (`es_valida`) no necesita ciclo propio: simplemente **combina** los resultados de las funciones auxiliares con `and`. Python evalúa un `and` de izquierda a derecha y se detiene apenas una condición es falsa (*short-circuit*), así que conviene poner primero la condición más barata de calcular (la longitud) para descartar rápido los casos obvios.

In [2]:
def tiene_mayuscula(texto: str) -> bool:
    for caracter in texto:
        if caracter.isupper():
            return True
    return False


def tiene_digito(texto: str) -> bool:
    for caracter in texto:
        if caracter.isdigit():
            return True
    return False


def es_valida(contraseña: str, longitud_minima: int = 8) -> bool:
    return (
        len(contraseña) >= longitud_minima
        and tiene_mayuscula(contraseña)
        and tiene_digito(contraseña)
    )


print(es_valida("abc123"))     # False (muy corta y sin mayúscula)
print(es_valida("Abcdefg1"))   # True


False
True


**Complejidad:** en el peor caso (ninguna regla se cumple, o se cumple hasta el último carácter) cada función auxiliar recorre toda la cadena: **O(n) en tiempo**, con `n = len(contraseña)`. Como `es_valida` llama a ambas una vez, sigue siendo O(n) (una constante de funciones auxiliares no cambia el orden). En **espacio** es **O(1)**: no se crean estructuras nuevas proporcionales al tamaño de la cadena.

## Ejercicio 3: Ciclo anidado + función auxiliar: tabla de multiplicar

### Enunciado

Escribe una función auxiliar `producto(a, b)` que devuelva el producto de dos números, y una función `imprimir_tabla(hasta)` que imprima la tabla de multiplicar del 1 al `hasta` (una fila por número, productos separados por tabulador), usando `producto` dentro del ciclo anidado en vez de escribir `fila * columna` directamente.

### Modelo de pensamiento

1. Aunque `fila * columna` es una operación trivial, aislarla en su propia función (`producto`) practica una idea central de este taller: el cuerpo de un ciclo (incluso uno anidado) puede ser una **llamada a función** en vez de una expresión directa — el mismo patrón que vas a necesitar cuando el cálculo interno sea más complejo que una simple multiplicación.
2. La estructura del ciclo anidado no cambia respecto al Taller 2: el ciclo externo controla la fila, el interno recorre las columnas de esa fila. Lo único nuevo es que, en cada posición, en vez de calcular directamente escribes `producto(fila, columna)`.
3. Esto también deja el código más legible: quien lea `texto_fila += f"{producto(fila, columna)}\t"` entiende de inmediato qué se está calculando, sin tener que interpretar una expresión aritmética suelta dentro del ciclo.

In [3]:
def producto(a: int, b: int) -> int:
    return a * b


def imprimir_tabla(hasta: int) -> None:
    for fila in range(1, hasta + 1):
        texto_fila = ""
        for columna in range(1, hasta + 1):
            texto_fila += f"{producto(fila, columna)}\t"
        print(texto_fila)


imprimir_tabla(5)


1	2	3	4	5	
2	4	6	8	10	
3	6	9	12	15	
4	8	12	16	20	
5	10	15	20	25	


**Complejidad:** hay dos ciclos anidados, cada uno de tamaño `h` (el parámetro `hasta`), y `producto` es O(1) → **O(h²) en tiempo**. En **espacio**, cada fila construye un string temporal de largo proporcional a `h` (que se descarta al imprimir e iniciar la siguiente fila) → **O(h)** de espacio auxiliar, no O(h²), porque no se guardan todas las filas a la vez.

## Ejercicio 4: Encontrar todas las posiciones (no solo la primera)

### Enunciado

En el Taller 2 buscaste la **primera** posición de un objetivo en una lista, cortando el ciclo con `break` apenas lo encontrabas. Ahora escribe `buscar_todas(lista, objetivo)` que devuelva **todas** las posiciones donde aparece el objetivo (una lista de índices, vacía si no aparece).

### Modelo de pensamiento

1. La diferencia clave con la búsqueda de la primera posición es que aquí **no puedes cortar el ciclo** apenas encuentres una coincidencia — necesitas seguir revisando el resto de la lista, porque puede haber más apariciones. El `break` desaparece del diseño.
2. En vez de una variable "bandera" que se sobreescribe (como `posicion_encontrada = indice`), ahora acumulas en una **lista**: cada vez que la condición se cumple, agregas el índice actual con `append`. Es el mismo patrón de "acumulador", solo que el acumulador es una colección en vez de un solo valor.
3. Como sí necesitas el índice (no solo el valor), recorres con `range(len(lista))`, igual que en el ejercicio original.

In [4]:
def buscar_todas(lista: list[int], objetivo: int) -> list[int]:
    posiciones: list[int] = []
    for indice in range(len(lista)):
        if lista[indice] == objetivo:
            posiciones.append(indice)
    return posiciones


print(buscar_todas([4, 8, 4, 15, 4, 23], 4))   # [0, 2, 4]
print(buscar_todas([4, 8, 15, 16, 23], 99))    # []


[0, 2, 4]
[]


**Complejidad:** a diferencia de la búsqueda de la primera posición (que en el mejor caso corta temprano), aquí **siempre** se recorre la lista completa porque no hay corte anticipado posible → **O(n) en tiempo**, con `n = len(lista)`, tanto en el mejor como en el peor caso. En **espacio** es **O(k)**, donde `k` es la cantidad de apariciones del objetivo (en el peor caso, `k = n` si todos los elementos son iguales al objetivo).

## Ejercicio 5: Composición de funciones: primos

### Enunciado

Escribe `es_primo(n)`, que determine si `n` es primo revisando divisores solo hasta `√n` (no hasta `n`). Luego escribe `primos_hasta(limite)`, que use un ciclo para llamar a `es_primo` sobre cada candidato entre 2 y `limite`, devolviendo la lista de los primos encontrados.

### Modelo de pensamiento

1. Para revisar si `n` tiene algún divisor, basta con probar hasta `√n`: si `n` tuviera un divisor mayor que su raíz, necesariamente tendría también uno menor que la acompaña (`n = a × b`, y si ambos fueran mayores que `√n`, el producto sería mayor que `n`). Esto es lo que hace que `es_primo` sea más rápido que revisar uno por uno hasta `n`.
2. `primos_hasta` no repite esa lógica: **reutiliza** `es_primo` dentro de su propio ciclo. Esta es la idea central del taller — descomponer en funciones que cooperan, en vez de escribir todo en un solo bloque.
3. Para analizar la complejidad de `primos_hasta` hay que pensar en dos niveles: cuántas veces se llama a `es_primo` (`limite - 1` veces) y cuánto cuesta **cada** llamada (hasta `√candidato`, que en el peor caso es `√limite`). El costo total combina ambos niveles.

In [5]:
def es_primo(n: int) -> bool:
    if n < 2:
        return False
    for divisor in range(2, int(n ** 0.5) + 1):
        if n % divisor == 0:
            return False
    return True


def primos_hasta(limite: int) -> list[int]:
    primos: list[int] = []
    for candidato in range(2, limite + 1):
        if es_primo(candidato):
            primos.append(candidato)
    return primos


print(primos_hasta(30))  # [2, 3, 5, 7, 11, 13, 17, 19, 23, 29]


[2, 3, 5, 7, 11, 13, 17, 19, 23, 29]


**Complejidad:** `es_primo(n)` cuesta **O(√n) en tiempo** y **O(1) en espacio** (no usa estructuras auxiliares). `primos_hasta(limite)` llama a `es_primo` una vez por cada candidato entre 2 y `limite`, así que el costo total en tiempo es **O(limite × √limite)**. En espacio, aparte de la constante de cada llamada, guarda la lista de primos encontrados: **O(p)**, donde `p` es la cantidad de primos hasta `limite`.

## Ejercicio 6: `while` controlado por una función auxiliar

### Enunciado

Retoma el ejercicio de ahorro del Taller 2. Ahora, en vez de escribir la comparación directamente en la condición del `while`, escribe una función auxiliar `meta_alcanzada(saldo, meta) -> bool`, y usa esa función dentro de `meses_para_meta(depositos, meta)` para decidir cuándo detener el ciclo.

### Modelo de pensamiento

1. Aunque `saldo < meta` cabría perfectamente escrita directamente en la condición del `while`, aislarla en una función con nombre (`meta_alcanzada`) hace que la condición del ciclo se lea casi como una oración: "mientras la meta no esté alcanzada y todavía haya depósitos, sigue sumando". Es la misma idea del Ejercicio 3: envolver una comparación simple en una función con nombre claro.
2. El `while` sigue necesitando **dos** motivos para detenerse (meta alcanzada, o depósitos agotados) — eso no cambia respecto al Taller 2; lo que cambia es que ahora una de esas dos condiciones vive en una función aparte, no escrita directamente en la línea del `while`.
3. Como antes, si el ciclo termina porque se acabaron los depósitos sin alcanzar la meta, ese es el caso especial que debe devolver -1. Revisa cuál de los dos motivos ocurrió llamando otra vez a `meta_alcanzada` después del ciclo, en vez de repetir la comparación escrita a mano.

In [6]:
def meta_alcanzada(saldo: float, meta: float) -> bool:
    return saldo >= meta


def meses_para_meta(depositos: list[float], meta: float) -> int:
    saldo = 0.0
    indice = 0
    while not meta_alcanzada(saldo, meta) and indice < len(depositos):
        saldo += depositos[indice]
        indice += 1

    if meta_alcanzada(saldo, meta):
        return indice
    else:
        return -1


print(meses_para_meta([100, 150, 200, 300], 400))  # 3
print(meses_para_meta([100, 100], 1000))            # -1


3
-1


**Complejidad:** el `while` da como máximo `len(depositos)` vueltas, y cada vuelta hace O(1) de trabajo (una suma y una llamada a `meta_alcanzada`, que también es O(1)) → **O(d) en tiempo**, con `d` el número de depósitos disponibles. En **espacio** es **O(1)**: solo se mantienen `saldo` e `indice`, sin estructuras que crezcan con `d`.

## Ejercicio 7: Resumen de transacciones (reto)

### Enunciado

Recibes una lista de transacciones, cada una un diccionario con las llaves `"tipo"` (`"ingreso"` o `"gasto"`), `"monto"` (float) y `"categoria"` (str). Escribe funciones que, combinadas, calculen: el total de ingresos, el total de gastos, el balance (ingresos − gastos), y un diccionario con el gasto total por categoría (solo contando gastos). Descompón el problema en al menos dos funciones auxiliares en vez de resolverlo todo en una sola función gigante.

### Modelo de pensamiento

1. Antes de escribir código, identifica que hay **dos preguntas distintas** sobre los mismos datos: "¿cuánto en total?" (un número) y "¿cuánto por categoría?" (un diccionario). Tratarlas como dos funciones separadas, cada una con su propio ciclo y su propio acumulador, es más claro que mezclar ambas responsabilidades en un solo bloque.
2. `total_por_tipo` reutiliza el patrón de "sumar solo lo que cumple una condición" del Taller 2 (como `contar_aprobados`, pero sumando montos en vez de contar), y recibe el tipo (`"ingreso"` o `"gasto"`) como parámetro para no repetir la función dos veces.
3. `gastos_por_categoria` combina ese mismo filtro con un **diccionario acumulador**: la clave es la categoría y el valor es la suma acumulada. La parte nueva es que, si la categoría todavía no está en el diccionario, hay que inicializarla en 0 antes de sumar — de lo contrario `diccionario[categoria] += monto` lanzaría un `KeyError` la primera vez que aparece esa categoría. `dict.get(categoria, 0.0)` resuelve ese caso de forma compacta.
4. Finalmente, una función `resumen_transacciones` compone las anteriores y arma el resultado completo, llamando a cada función auxiliar con los parámetros que necesita — es la capa que junta las piezas, sin repetir ningún ciclo.

In [7]:
from typing import Any

Transaccion = dict[str, Any]


def total_por_tipo(transacciones: list[Transaccion], tipo: str) -> float:
    total = 0.0
    for transaccion in transacciones:
        if transaccion["tipo"] == tipo:
            total += transaccion["monto"]
    return total


def gastos_por_categoria(transacciones: list[Transaccion]) -> dict[str, float]:
    totales: dict[str, float] = {}
    for transaccion in transacciones:
        if transaccion["tipo"] == "gasto":
            categoria = transaccion["categoria"]
            totales[categoria] = totales.get(categoria, 0.0) + transaccion["monto"]
    return totales


def resumen_transacciones(transacciones: list[Transaccion]) -> dict[str, Any]:
    ingresos = total_por_tipo(transacciones, "ingreso")
    gastos = total_por_tipo(transacciones, "gasto")
    return {
        "ingresos": ingresos,
        "gastos": gastos,
        "balance": ingresos - gastos,
        "gastos_por_categoria": gastos_por_categoria(transacciones),
    }


transacciones = [
    {"tipo": "ingreso", "monto": 2000.0, "categoria": "salario"},
    {"tipo": "gasto", "monto": 300.0, "categoria": "comida"},
    {"tipo": "gasto", "monto": 150.0, "categoria": "transporte"},
    {"tipo": "gasto", "monto": 100.0, "categoria": "comida"},
]

resumen = resumen_transacciones(transacciones)
print(resumen)


{'ingresos': 2000.0, 'gastos': 550.0, 'balance': 1450.0, 'gastos_por_categoria': {'comida': 400.0, 'transporte': 150.0}}


**Complejidad:** `total_por_tipo` y `gastos_por_categoria` recorren la lista de transacciones una vez cada una → O(n) cada una, con `n` = número de transacciones. `resumen_transacciones` llama a `total_por_tipo` dos veces y a `gastos_por_categoria` una vez, es decir, un múltiplo constante de recorridos sobre `n` → sigue siendo **O(n) en tiempo** en total (las constantes no cambian el orden de magnitud). En **espacio**, lo único que crece con los datos es el diccionario `gastos_por_categoria`, de tamaño **O(c)**, donde `c` es el número de categorías distintas (en el peor caso, `c = n` si cada transacción tiene una categoría única).